# 2.4 — Đánh giá validation và phân tích lỗi DL

So sánh MLP baseline, TabNet baseline, MLP weighted và MLP embedding bằng cùng tập validation seed 9. ROC-AUC là metric chính; Average Precision là metric phụ. Chỉ dùng artifact validation, không đọc test labels hoặc public leaderboard.

## 2.4.1 — Nạp metrics và dự đoán validation

Đính kèm output Kaggle của task 2.1, 2.2 và 2.3 làm input. Notebook kiểm tra đủ bốn file dự đoán, cùng membership ID và cùng nhãn validation trước khi so sánh.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/task_2_4_validation_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ID_COL = "id"
TARGET = "is_fraud"


def find_unique_file(filename):
    matches = list(INPUT_ROOT.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected exactly one file {filename}; found {len(matches)}.")
    return matches[0]


def find_task_metrics(task_name):
    matches = []
    for path in INPUT_ROOT.rglob("metrics.json"):
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if payload.get("task") == task_name:
            matches.append(payload)
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected exactly one metrics.json for {task_name}; found {len(matches)}.")
    return matches[0]


metrics_21 = find_task_metrics("2.1 MLP baseline")
metrics_22 = find_task_metrics("2.2 TabNet baseline")
metrics_23 = find_task_metrics("2.3 MLP imbalance experiments")

prediction_paths = {
    "MLP baseline": find_unique_file("validation_predictions_seed9.csv"),
    "TabNet baseline": find_unique_file("validation_predictions_tabnet_seed9.csv"),
    "MLP weighted": find_unique_file("validation_predictions_mlp_weighted_seed9.csv"),
    "MLP embedding": find_unique_file("validation_predictions_mlp_embedding_seed9.csv"),
}
predictions = {name: pd.read_csv(path) for name, path in prediction_paths.items()}

reference = predictions["MLP baseline"]
reference_ids = reference[ID_COL].to_numpy()
reference_y = reference[TARGET].to_numpy(dtype=np.int64)
for name, frame in list(predictions.items()):
    required = {ID_COL, TARGET, "fraud_probability"}
    if not required.issubset(frame.columns):
        raise ValueError(f"{name}: missing required prediction columns.")
    if frame[ID_COL].duplicated().any() or set(frame[ID_COL]) != set(reference_ids):
        raise ValueError(f"{name}: validation membership differs by id.")
    frame = frame.set_index(ID_COL).loc[reference_ids].reset_index()
    if not np.array_equal(frame[TARGET].to_numpy(dtype=np.int64), reference_y):
        raise ValueError(f"{name}: validation labels differ after aligning by id.")
    probabilities = frame["fraud_probability"].to_numpy(dtype=np.float64)
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError(f"{name}: probabilities are not finite or fall outside [0, 1].")
    predictions[name] = frame
print("Validation rows shared by all models:", len(reference))
print("Fraud rate in validation:", round(float(reference_y.mean()), 6))
print("Loaded sources:", {name: str(path) for name, path in prediction_paths.items()})

## 2.4.2 — So sánh metric và learning curves

Tính lại ROC-AUC/PR-AUC từ xác suất và nhãn validation đã kiểm tra. Thời gian chỉ hiện khi artifact ghi nhận; nếu baseline 2.1 không lưu thời gian thì để trống. Learning curves lấy từ lịch sử epoch của từng mô hình.

In [ ]:
experiment_23 = {item["name"]: item for item in metrics_23["experiments"] if "name" in item}
metadata_by_model = {
    "MLP baseline": {
        "best_epoch": metrics_21.get("best_epoch"),
        "training_seconds": metrics_21.get("training_seconds"),
    },
    "TabNet baseline": {
        "best_epoch": None if metrics_22.get("best_epoch") is None else int(metrics_22["best_epoch"]) + 1,  # TabNet epoch index -> epoch count
        "training_seconds": metrics_22.get("training_seconds"),
    },
    "MLP weighted": experiment_23.get("MLP weighted", {}),
    "MLP embedding": experiment_23.get("MLP embedding", {}),
}

rows = []
for name, frame in predictions.items():
    probability = frame["fraud_probability"].to_numpy(dtype=np.float64)
    extra = metadata_by_model[name]
    rows.append({
        "model": name,
        "validation_roc_auc": float(roc_auc_score(reference_y, probability)),
        "validation_average_precision": float(average_precision_score(reference_y, probability)),
        "training_seconds": extra.get("training_seconds"),
        "best_epoch": extra.get("best_epoch"),
    })
comparison = pd.DataFrame(rows).sort_values(
    ["validation_roc_auc", "validation_average_precision"],
    ascending=False,
).reset_index(drop=True)
comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)
display(comparison)

history_paths = list(INPUT_ROOT.rglob("training_history.csv"))
history_expectations = {
    "MLP baseline": lambda p: "task_2_1_mlp" in p.as_posix(),
    "TabNet baseline": lambda p: "task_2_2_tabnet" in p.as_posix(),
    "MLP weighted": lambda p: p.parent.name == "mlp_weighted",
    "MLP embedding": lambda p: p.parent.name == "mlp_embedding",
}
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, matcher in history_expectations.items():
    matches = [path for path in history_paths if matcher(path)]
    if len(matches) != 1:
        print(f"Learning history for {name}: expected 1, found {len(matches)}.")
        continue
    history = pd.read_csv(matches[0])
    auc_col = next((col for col in ["val_roc_auc", "valid_auc", "validation_roc_auc"] if col in history), None)
    loss_col = next((col for col in ["train_loss", "loss"] if col in history), None)
    if auc_col:
        axes[0].plot(history["epoch"] if "epoch" in history else np.arange(1, len(history) + 1), history[auc_col], marker=".", label=name)
    if loss_col:
        axes[1].plot(history["epoch"] if "epoch" in history else np.arange(1, len(history) + 1), history[loss_col], marker=".", label=name)
axes[0].set(title="Validation ROC-AUC by epoch", xlabel="Epoch", ylabel="ROC-AUC")
axes[1].set(title="Training loss by epoch", xlabel="Epoch", ylabel="Loss")
for axis in axes:
    axis.legend()
    axis.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "learning_curves_comparison.png", dpi=150)
plt.show()
plt.close(fig)

## 2.4.3–2.4.4 — Phân phối xác suất và lỗi tại threshold 0.5

Vẽ ROC curves và phân phối xác suất. Với mô hình được chọn, dùng threshold 0.5 chỉ để đếm false positives/false negatives và diễn giải lỗi. Threshold này không dùng để chấm ROC-AUC hoặc tạo Kaggle submission.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, frame in predictions.items():
    probability = frame["fraud_probability"].to_numpy(dtype=np.float64)
    fpr, tpr, _ = roc_curve(reference_y, probability)
    auc_value = roc_auc_score(reference_y, probability)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={auc_value:.5f})")
    axes[1].hist(probability, bins=40, alpha=0.35, density=True, label=name)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set(title="Validation ROC curves", xlabel="False positive rate", ylabel="True positive rate")
axes[1].set(title="Validation probability distributions", xlabel="Predicted fraud probability", ylabel="Density")
axes[0].legend()
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "roc_and_probability_distributions.png", dpi=150)
plt.show()
plt.close(fig)

selected_name = str(comparison.iloc[0]["model"])
selected_frame = predictions[selected_name]
selected_probability = selected_frame["fraud_probability"].to_numpy(dtype=np.float64)
threshold = 0.5
predicted_label = (selected_probability >= threshold).astype(np.int64)
false_positive = int(((predicted_label == 1) & (reference_y == 0)).sum())
false_negative = int(((predicted_label == 0) & (reference_y == 1)).sum())
true_positive = int(((predicted_label == 1) & (reference_y == 1)).sum())
true_negative = int(((predicted_label == 0) & (reference_y == 0)).sum())
precision = true_positive / max(1, true_positive + false_positive)
recall = true_positive / max(1, true_positive + false_negative)

error_analysis = {
    "model": selected_name,
    "threshold_for_error_interpretation_only": threshold,
    "true_positive": true_positive,
    "true_negative": true_negative,
    "false_positive": false_positive,
    "false_negative": false_negative,
    "precision_at_threshold": precision,
    "recall_at_threshold": recall,
}
print(json.dumps(error_analysis, indent=2))

## 2.4.5–2.4.7 — Chọn mô hình và ghi quyết định

Chọn theo ROC-AUC validation; Average Precision là tie-break. Cross-validation không chạy trong lượt này để giữ holdout chung cho so sánh và dành compute cho final refit. Đây là giới hạn cần nêu trong báo cáo.

In [ ]:
selected_row = comparison.iloc[0].to_dict()
decision = {
    "selected_model": selected_name,
    "selection_rule": "Highest shared-validation ROC-AUC; Average Precision as tie-break",
    "validation_metrics": {
        "roc_auc": float(selected_row["validation_roc_auc"]),
        "average_precision": float(selected_row["validation_average_precision"]),
    },
    "best_epoch": None if pd.isna(selected_row["best_epoch"]) else int(selected_row["best_epoch"]),
    "training_seconds": None if pd.isna(selected_row["training_seconds"]) else float(selected_row["training_seconds"]),
    "error_analysis": error_analysis,
    "cross_validation": "Not run; single fixed shared holdout used for comparable model selection.",
    "test_data_used_for_model_selection": False,
    "public_leaderboard_used_for_model_selection": False,
}
with open(OUTPUT_DIR / "selected_model.json", "w", encoding="utf-8") as file:
    json.dump(decision, file, indent=2)

summary_lines = [
    "# Task 2.4 - Validation analysis",
    "",
    f"Selected candidate: {selected_name}",
    f"Validation ROC-AUC: {selected_row['validation_roc_auc']:.6f}",
    f"Validation Average Precision: {selected_row['validation_average_precision']:.6f}",
    f"False positives / false negatives at threshold 0.5 (interpretation only): {false_positive} / {false_negative}",
    "",
    "Selection used only the shared seed-9 validation split. Cross-validation was not run; the result reflects one holdout split.",
]
(OUTPUT_DIR / "analysis_summary.md").write_text("\n".join(summary_lines), encoding="utf-8")
print("Selected model:", selected_name)
print("Saved comparison, plots, error analysis and selected_model.json to:", OUTPUT_DIR)